In [1]:
import numpy as np
import sympy as sp

In [4]:
class QuadraticOptimizer:

    def __init__(self, X, Y):
        self.X = np.array(X, dtype=float)
        self.Y = np.array(Y, dtype=float).reshape(-1, 1)

        self.n, self.m = self.X.shape
        self.k = self.m + 1

        # 1. Xây dựng ma trận Q
        self.Q = np.column_stack([
            np.ones(self.n),
            self.X
        ])

        # 2. Tính A, b, c
        self.A = self.Q.T @ self.Q
        self.b = -self.Y.T @ self.Q
        self.c = self.Y.T @ self.Y

        # 3. Khởi tạo các biến p0, p1, ..., pm
        self.P = sp.Matrix(
            sp.symbols(f'p0:{self.k}')
        )

        # Chuyển sang SymPy
        A_sym = sp.Matrix(self.A)
        b_sym = sp.Matrix(self.b)
        c_sym = sp.Matrix(self.c)

        # 4. Xây dựng hàm mục tiêu
        self.f = sp.expand(
            (
                sp.Rational(1, 2)
                * self.P.T * A_sym * self.P
                + b_sym * self.P
                + sp.Rational(1, 2) * c_sym
            )[0]
        )

        # 5. Tính Gradient
        self.gradient = sp.Matrix([
            sp.diff(self.f, p)
            for p in self.P
        ])

        # 6. Tính Hessian
        self.hessian = sp.hessian(
            self.f, list(self.P)
        )

    # Hiển thị A, b, c
    def show_ABC(self):

        print("\n========== MATRIX A ==========")
        sp.pprint(sp.Matrix(self.A))

        print("\n========== VECTOR b ==========")
        sp.pprint(sp.Matrix(self.b))

        print("\n========== CONSTANT c ==========")
        sp.pprint(sp.Matrix(self.c))

    # Hiển thị hàm mục tiêu
    def show_function(self):

        print("\n========== OBJECTIVE FUNCTION ==========")
        print("f(P) = 1/2 * P.T * A * P + b * P + 1/2 * c")
        sp.pprint(self.f)

    # Hiển thị Gradient
    def show_gradient(self):

        print("\n========== GRADIENT ==========")
        sp.pprint(self.gradient)

    # Hiển thị Hessian
    def show_hessian(self):

        print("\n========== HESSIAN ==========")
        sp.pprint(self.hessian)

    # Tính các định thức con chính đầu
    def show_determinants(self):

        print("\n========== HESSIAN DETERMINANTS ==========")

        for i in range(1, self.k + 1):

            sub_matrix = self.hessian[:i, :i]

            determinant = sub_matrix.det()

            print(f"\nD{i} =")
            sp.pprint(determinant)

            print("Value =", float(determinant))

    # Kiểm tra tính xác định dương
    def check_positive_definite(self):

        print("\n========== CHECK HESSIAN ==========")

        determinants = []

        for i in range(1, self.k + 1):

            determinant = self.hessian[:i, :i].det()

            determinants.append(float(determinant))

        if all(d > 1e-10 for d in determinants):

            print("Hessian is POSITIVE DEFINITE")
            print("=> Unique global minimum")

        elif all(d >= -1e-10 for d in determinants):

            eigenvalues = np.linalg.eigvalsh(self.A)

            if np.all(eigenvalues >= -1e-10):
                print("Hessian is POSITIVE SEMIDEFINITE")
                print("=> Global minimum exists, may not be unique")
            else:
                print("Hessian is not positive definite")

        else:
            print("Hessian is NOT positive definite")

    # Tìm nghiệm tối ưu
    def solve(self):

        print("\n========== OPTIMAL SOLUTION ==========")

        # Giải hệ A*P = -b.T
        P_opt, residuals, rank, singular_values = (
            np.linalg.lstsq(
                self.A,
                -self.b.T,
                rcond=None
            )
        )

        # Hiển thị nghiệm
        for i, value in enumerate(P_opt.flatten()):

            print(f"p{i} = {value:.10f}")

        # Tính giá trị hàm mục tiêu
        f_min = (
            0.5 * P_opt.T @ self.A @ P_opt
            + self.b @ P_opt
            + 0.5 * self.c
        )

        print("\nf_min =", float(f_min[0, 0]))

        # Gradient tại nghiệm
        gradient_opt = (
            self.A @ P_opt + self.b.T
        )

        print("\nGradient at optimum:")
        print(gradient_opt)

        print("\nMatrix rank =", rank)

        return P_opt




In [5]:
# ==========================================
# MAIN
# ==========================================

if __name__ == "__main__":

    # Dữ liệu đề bài
    X = [
        [1],
        [2],
        [3],
        [4],
        [5],
        [6],
        [7],
        [8],
        [9],
        [10]
    ]

    Y = [
        1.8,
        2.5,
        3.2,
        4.1,
        4.8,
        5.5,
        6.8,
        7.5,
        9.3,
        12.5
    ]

    optimizer = QuadraticOptimizer(X, Y)

    optimizer.show_ABC()

    optimizer.show_function()

    optimizer.show_gradient()

    optimizer.show_hessian()

    optimizer.show_determinants()

    optimizer.check_positive_definite()

    optimizer.solve()



========== MATRIX A ==========
⎡10.0  55.0 ⎤
⎢           ⎥
⎣55.0  385.0⎦

========== VECTOR b ==========
[-58.0  -406.1]

========== CONSTANT c ==========
[435.06]

========== OBJECTIVE FUNCTION ==========
f(P) = 1/2 * P.T * A * P + b * P + 1/2 * c
      2                                  2                    
5.0⋅p₀  + 55.0⋅p₀⋅p₁ - 58.0⋅p₀ + 192.5⋅p₁  - 406.1⋅p₁ + 217.53

========== GRADIENT ==========
⎡ 10.0⋅p₀ + 55.0⋅p₁ - 58.0 ⎤
⎢                          ⎥
⎣55.0⋅p₀ + 385.0⋅p₁ - 406.1⎦

========== HESSIAN ==========
⎡10.0  55.0 ⎤
⎢           ⎥
⎣55.0  385.0⎦

========== HESSIAN DETERMINANTS ==========

D1 =
10.0000000000000
Value = 10.0

D2 =
825.000000000000
Value = 825.0

========== CHECK HESSIAN ==========
Hessian is POSITIVE DEFINITE
=> Unique global minimum

========== OPTIMAL SOLUTION ==========
p0 = -0.0066666667
p1 = 1.0557575758

f_min = 3.3517575757576026

Gradient at optimum:
[[1.42108547e-14]
 [1.70530257e-13]]

Matrix rank = 2


In [6]:


if __name__ == "__main__":

    # Dữ liệu đề bài
    X = [
        [30],
        [32],
        [34],
        [36],
        [38],
    ]

    Y = [
        36.2,
        36.2,
        37.8,
        39.7,
        43.7,
    ]

    optimizer = QuadraticOptimizer(X, Y)

    optimizer.show_ABC()

    optimizer.show_function()

    optimizer.show_gradient()

    optimizer.show_hessian()

    optimizer.show_determinants()

    optimizer.check_positive_definite()

    optimizer.solve()


========== MATRIX A ==========
⎡ 5.0   170.0 ⎤
⎢             ⎥
⎣170.0  5820.0⎦

========== VECTOR b ==========
[-193.6  -6619.4]

========== CONSTANT c ==========
[7535.5]

========== OBJECTIVE FUNCTION ==========
f(P) = 1/2 * P.T * A * P + b * P + 1/2 * c
      2                                     2                      
2.5⋅p₀  + 170.0⋅p₀⋅p₁ - 193.6⋅p₀ + 2910.0⋅p₁  - 6619.4⋅p₁ + 3767.75

========== GRADIENT ==========
⎡  5.0⋅p₀ + 170.0⋅p₁ - 193.6  ⎤
⎢                             ⎥
⎣170.0⋅p₀ + 5820.0⋅p₁ - 6619.4⎦

========== HESSIAN ==========
⎡ 5.0   170.0 ⎤
⎢             ⎥
⎣170.0  5820.0⎦

========== HESSIAN DETERMINANTS ==========

D1 =
5.00000000000000
Value = 5.0

D2 =
200.000000000000
Value = 200.0

========== CHECK HESSIAN ==========
Hessian is POSITIVE DEFINITE
=> Unique global minimum

========== OPTIMAL SOLUTION ==========
p0 = 7.2700000000
p1 = 0.9250000000

f_min = 2.5414999999998145

Gradient at optimum:
[[-8.52651283e-14]
 [-1.81898940e-12]]

Matrix rank = 2


In [7]:


if __name__ == "__main__":

    # Dữ liệu đề bài
    X = [
        [100],
        [120],
        [140],
        [160],
        [180],
        [200],
    ]

    Y = [
        171, 194.9, 227.3, 255.7, 283, 309.3
    ]

    optimizer = QuadraticOptimizer(X, Y)

    optimizer.show_ABC()

    optimizer.show_function()

    optimizer.show_gradient()

    optimizer.show_hessian()

    optimizer.show_determinants()

    optimizer.check_positive_definite()

    optimizer.solve()


========== MATRIX A ==========
⎡ 6.0    900.0  ⎤
⎢               ⎥
⎣900.0  142000.0⎦

========== VECTOR b ==========
[-1441.2  -226022.0]

========== CONSTANT c ==========
[360030.28]

========== OBJECTIVE FUNCTION ==========
f(P) = 1/2 * P.T * A * P + b * P + 1/2 * c
      2                                       2                          
3.0⋅p₀  + 900.0⋅p₀⋅p₁ - 1441.2⋅p₀ + 71000.0⋅p₁  - 226022.0⋅p₁ + 180015.14

========== GRADIENT ==========
⎡   6.0⋅p₀ + 900.0⋅p₁ - 1441.2    ⎤
⎢                                 ⎥
⎣900.0⋅p₀ + 142000.0⋅p₁ - 226022.0⎦

========== HESSIAN ==========
⎡ 6.0    900.0  ⎤
⎢               ⎥
⎣900.0  142000.0⎦

========== HESSIAN DETERMINANTS ==========

D1 =
6.00000000000000
Value = 6.0

D2 =
42000.0000000000
Value = 42000.0

========== CHECK HESSIAN ==========
Hessian is POSITIVE DEFINITE
=> Unique global minimum

========== OPTIMAL SOLUTION ==========
p0 = 29.3000000002
p1 = 1.4060000000

f_min = 8.094000000011874

Gradient at optimum:
[[4.88853402e-11]
 [5.